--- Day 17: Chronospatial Computer ---
The Historians push the button on their strange device, but this time, you all just feel like you're falling.

"Situation critical", the device announces in a familiar voice. "Bootstrapping process failed. Initializing debugger...."

The small handheld device suddenly unfolds into an entire computer! The Historians look around nervously before one of them tosses it to you.

This seems to be a 3-bit computer: its program is a list of 3-bit numbers (0 through 7), like 0,1,2,3. The computer also has three registers named A, B, and C, but these registers aren't limited to 3 bits and can instead hold any integer.

The computer knows eight instructions, each identified by a 3-bit number (called the instruction's opcode). Each instruction also reads the 3-bit number after it as an input; this is called its operand.

A number called the instruction pointer identifies the position in the program from which the next opcode will be read; it starts at 0, pointing at the first 3-bit number in the program. Except for jump instructions, the instruction pointer increases by 2 after each instruction is processed (to move past the instruction's opcode and its operand). If the computer tries to read an opcode past the end of the program, it instead halts.

So, the program 0,1,2,3 would run the instruction whose opcode is 0 and pass it the operand 1, then run the instruction having opcode 2 and pass it the operand 3, then halt.

There are two types of operands; each instruction specifies the type of its operand. The value of a literal operand is the operand itself. For example, the value of the literal operand 7 is the number 7. The value of a combo operand can be found as follows:

Combo operands 0 through 3 represent literal values 0 through 3.
Combo operand 4 represents the value of register A.
Combo operand 5 represents the value of register B.
Combo operand 6 represents the value of register C.
Combo operand 7 is reserved and will not appear in valid programs.
The eight instructions are as follows:

The adv instruction (opcode 0) performs division. The numerator is the value in the A register. The denominator is found by raising 2 to the power of the instruction's combo operand. (So, an operand of 2 would divide A by 4 (2^2); an operand of 5 would divide A by 2^B.) The result of the division operation is truncated to an integer and then written to the A register.

The bxl instruction (opcode 1) calculates the bitwise XOR of register B and the instruction's literal operand, then stores the result in register B.

The bst instruction (opcode 2) calculates the value of its combo operand modulo 8 (thereby keeping only its lowest 3 bits), then writes that value to the B register.

The jnz instruction (opcode 3) does nothing if the A register is 0. However, if the A register is not zero, it jumps by setting the instruction pointer to the value of its literal operand; if this instruction jumps, the instruction pointer is not increased by 2 after this instruction.

The bxc instruction (opcode 4) calculates the bitwise XOR of register B and register C, then stores the result in register B. (For legacy reasons, this instruction reads an operand but ignores it.)

The out instruction (opcode 5) calculates the value of its combo operand modulo 8, then outputs that value. (If a program outputs multiple values, they are separated by commas.)

The bdv instruction (opcode 6) works exactly like the adv instruction except that the result is stored in the B register. (The numerator is still read from the A register.)

The cdv instruction (opcode 7) works exactly like the adv instruction except that the result is stored in the C register. (The numerator is still read from the A register.)

Here are some examples of instruction operation:

If register C contains 9, the program 2,6 would set register B to 1.
If register A contains 10, the program 5,0,5,1,5,4 would output 0,1,2.
If register A contains 2024, the program 0,1,5,4,3,0 would output 4,2,5,6,7,7,7,7,3,1,0 and leave 0 in register A.
If register B contains 29, the program 1,7 would set register B to 26.
If register B contains 2024 and register C contains 43690, the program 4,0 would set register B to 44354.
The Historians' strange device has finished initializing its debugger and is displaying some information about the program it is trying to run (your puzzle input). For example:

Register A: 729
Register B: 0
Register C: 0

Program: 0,1,5,4,3,0
Your first task is to determine what the program is trying to output. To do this, initialize the registers to the given values, then run the given program, collecting any output produced by out instructions. (Always join the values produced by out instructions with commas.) After the above program halts, its final output will be 4,6,3,5,6,3,5,2,1,0.

Using the information provided by the debugger, initialize the registers to the given values, then run the program. Once it halts, what do you get if you use commas to join the values it output into a single string?

In [17]:
sample_input = """
Register A: 729
Register B: 0
Register C: 0

Program: 0,1,5,4,3,0"""


puzzle_input = """
Register A: 64854237
Register B: 0
Register C: 0

Program: 2,4,1,1,7,5,1,5,4,0,5,5,0,3,3,0"""

In [18]:
class Program:
    def __init__(self, a, b, c, prog):
        self.a = a
        self.b = b
        self.c = c
        self.prog = prog
        self.pointer = 0

    def combo(self, op):
        if 0 <= op <= 3:
            return op
        elif op == 4:
            return self.a
        elif op == 5:
            return self.b
        elif op == 6:
            return self.c

    def adv(self, op):
        numerator = self.a
        denominator = 2 ** self.combo(op)
        self.a = int(numerator / denominator)

    def bxl(self, op):
        self.b = op ^ self.b

    def bst(self, op):
        self.b = self.combo(op) % 8

    def jnz(self, op):
        if self.a != 0:
            self.pointer = op - 2 # execute() will offset the -2 decrement

    def bxc(self, op):
        self.b = self.b ^ self.c

    def out(self, op, out=None):
        output = self.combo(op) % 8
        if out:
            return str(output)
        else:
            print(output, end=",")
        
    def bdv(self, op):
        numerator = self.a
        denominator = 2 ** self.combo(op)
        self.b = int(numerator / denominator)

    def cdv(self, op):
        numerator = self.a
        denominator = 2 ** self.combo(op)
        self.c = int(numerator / denominator)

    def execute(self, out=None):
        instr = {0 : self.adv,
                1 : self.bxl,
                2 : self.bst,
                3 : self.jnz,
                4 : self.bxc,
                5 : self.out,
                6 : self.bdv,
                7 : self.cdv}
        try:
            opcode = self.prog[self.pointer]
            operand = self.prog[self.pointer + 1]
            if opcode == 5 and out:
                self.pointer += 2
                return self.out(operand, out)
            else:
                instr[opcode](operand)
                self.pointer += 2
        except IndexError:
            raise IndexError("Program halted")


In [19]:
# Sample input:
sample_prog = Program(729,0,0,[int(i) for i in "0,1,5,4,3,0".split(",")])
while True:
    sample_prog.execute()

4,6,3,5,6,3,5,2,1,0,

IndexError: Program halted

In [ ]:
# Puzzle input:
a = 64854237
b = c = 0
prog = "2,4,1,1,7,5,1,5,4,0,5,5,0,3,3,0"
prog = [int(i) for i in prog.split(",")]

In [ ]:
puzzle_prog = Program(a,b,c,prog)
while True:
    puzzle_prog.execute()

4,1,7,6,4,1,0,2,7,

IndexError: Program halted

Your puzzle answer was 4,1,7,6,4,1,0,2,7.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
Digging deeper in the device's manual, you discover the problem: this program is supposed to output another copy of the program! Unfortunately, the value in register A seems to have been corrupted. You'll need to find a new value to which you can initialize register A so that the program's output instructions produce an exact copy of the program itself.

For example:

Register A: 2024
Register B: 0
Register C: 0

Program: 0,3,5,4,3,0
This program outputs a copy of itself if register A is instead initialized to 117440. (The original initial value of register A, 2024, is ignored.)

What is the lowest positive initial value for register A that causes the program to output a copy of itself?

In [ ]:
"""
This is a much more interesting problem. I can think of roughly three approaches to this problem: We can bruteforce it, we can look for numerical patterns in the instructions that we can exploit, or we can see if we can reverse engineer the necessary steps from the point of halting backwards.

Here is our program we need to reproduce: "2,4,1,1,7,5,1,5,4,0,5,5,0,3,3,0"

Some observations:
The initial codeblock, "2,4,1,1,7,5,1,5,4,0,5,5,0,3" will always be executed first
After that, "3,0" will jump back to the beginning unless A is 0 (in which case the program halts). This makes the order of commands a whole lot more predictable:

2,4: b = a % 8
1,1: b = b ^ 1
7,5: c = a // 2**b
1,5: b = b ^ 5
4,0: b = b ^ c
5,5: print b mod 8
0,3: a = a // 8

We need to output a total of 16 characters, that means that on the 16th iteration a needs to be 0. Fortunately, a is only modified once and independenly of other variables. If a's initial value is below 8, the program will output one number. If it is between 8 and 64, it will output 2 and so on. So a value from [8**(x-1), 8**x) will output x numbers. Therefore, a's initial value is between 8**15 and 8**16

A further simplification: On each pass through the program, all other registers are effectively reset based on the new value of a. Therefore, we only need to look at which values of a produce which outputs. 
"""

In [ ]:
start = 10

for a in range(start, start + 1000):
    explore_prog = Program(a,b,c,prog)
    print(f" a {a}: out ", end="")
    while True:
        try:
            explore_prog.execute()
        except IndexError:
            print("")
            break

 a 10: out 7,4,
 a 11: out 5,4,
 a 12: out 0,4,
 a 13: out 1,4,
 a 14: out 2,4,
 a 15: out 3,4,
 a 16: out 4,6,
 a 17: out 4,6,
 a 18: out 4,6,
 a 19: out 3,6,
 a 20: out 0,6,
 a 21: out 0,6,
 a 22: out 2,6,
 a 23: out 3,6,
 a 24: out 0,7,
 a 25: out 4,7,
 a 26: out 5,7,
 a 27: out 1,7,
 a 28: out 0,7,
 a 29: out 0,7,
 a 30: out 2,7,
 a 31: out 3,7,
 a 32: out 4,0,
 a 33: out 4,0,
 a 34: out 2,0,
 a 35: out 7,0,
 a 36: out 1,0,
 a 37: out 3,0,
 a 38: out 2,0,
 a 39: out 3,0,
 a 40: out 0,1,
 a 41: out 4,1,
 a 42: out 3,1,
 a 43: out 5,1,
 a 44: out 1,1,
 a 45: out 3,1,
 a 46: out 2,1,
 a 47: out 3,1,
 a 48: out 4,2,
 a 49: out 4,2,
 a 50: out 0,2,
 a 51: out 3,2,
 a 52: out 1,2,
 a 53: out 2,2,
 a 54: out 2,2,
 a 55: out 3,2,
 a 56: out 0,3,
 a 57: out 4,3,
 a 58: out 1,3,
 a 59: out 1,3,
 a 60: out 1,3,
 a 61: out 2,3,
 a 62: out 2,3,
 a 63: out 3,3,
 a 64: out 4,0,4,
 a 65: out 4,0,4,
 a 66: out 6,0,4,
 a 67: out 7,0,4,
 a 68: out 2,0,4,
 a 69: out 5,0,4,
 a 70: out 2,0,4,
 a 71: out

In [ ]:
"""Having looked at some outputs, there seems to be a periodicity to the outputs that's interesting. Also, we can calculate around 100k programs in around 5 seconds.

We can look at that more later. But let's try to work backwards and narrow down the needed outputs.

Our last number is 0 and A needs to be between 0 and 7. Therefore, A needs to be 4 in the last round.

if A(16) = 4, A(15) // 8 = 4, therefore A(15) is between 32 and 32 + 7.

Our last 2 output numbers are 3,0. What value does A(15) need to be?
"""

In [30]:
start = 32
outputs = {}
for a in range(start, 39):
    explore_prog = Program(a,b,c,prog)
    print(f" a {a}: out ", end="")
    output = ""
    while True:
        try:
            if out := explore_prog.execute(out=True):
                output += out + ","
        except IndexError:
            output = output[:-1]
            print(output)
            outputs[a] = output
            break

 a 32: out 4,0
 a 33: out 4,0
 a 34: out 2,0
 a 35: out 7,0
 a 36: out 1,0
 a 37: out 3,0
 a 38: out 2,0


In [26]:
for a, out in outputs.items():
    if out == "3,3,0" or out == "3,0":
        print(a, out)

37 3,0
39 3,0


In [ ]:
"""So we can see that for the second to last iteration, a needs to be either 37 or 39. Let's try one more iteration by hand before automating the loop: 
If A(15) = 37, A 14 needs to be between 37 * 8 and 37 * 8 +7. Similarly, we can have 39 * 8 to 39 * 8 + 7"""

In [31]:
start = 37*8
outputs = {}
for a in range(start, start + 8):
    explore_prog = Program(a,b,c,prog)
    print(f" a {a}: out ", end="")
    output = ""
    while True:
        try:
            if out := explore_prog.execute(out=True):
                output += out + ","
        except IndexError:
            output = output[:-1]
            print(output)
            outputs[a] = output
            break

 a 296: out 0,3,0
 a 297: out 4,3,0
 a 298: out 3,3,0
 a 299: out 5,3,0
 a 300: out 1,3,0
 a 301: out 3,3,0
 a 302: out 0,3,0
 a 303: out 7,3,0


In [32]:
start = 39*8
outputs = {}
for a in range(start, start + 8):
    explore_prog = Program(a,b,c,prog)
    print(f" a {a}: out ", end="")
    output = ""
    while True:
        try:
            if out := explore_prog.execute(out=True):
                output += out + ","
        except IndexError:
            output = output[:-1]
            print(output)
            outputs[a] = output
            break

 a 312: out 0,3,0
 a 313: out 4,3,0
 a 314: out 1,3,0
 a 315: out 1,3,0
 a 316: out 1,3,0
 a 317: out 2,3,0
 a 318: out 0,3,0
 a 319: out 7,3,0


In [ ]:
"""Ok. iterating like this our path of values we have to check is actually incredibly narrow. Let's work on automating that loop"""

In [51]:
def find_a(prog, range_a):
    prog_str = ",".join((str(s) for s in prog))
    valid_a = set()
    for a in range_a:
        p = Program(a,0,0,prog)
        p_out = ""
        while True:
            try:
                if out := p.execute(out=True):
                    p_out += out + ","
            except IndexError:
                p_out = p_out[:-1]
                if prog_str.endswith(p_out):
                    valid_a.add(a)
                break
    return valid_a
find_a(prog, range(100))

{4, 37, 39}

In [60]:
range_a = range(1000)
partials = find_a(prog, range_a)
solutions = set()
while partials:
    partial_a = partials.pop()
    if 8 ** 15 <= partial_a <= 8 ** 16:
        solutions.add(partial_a)
    else:
        range_a = range(partial_a * 8, partial_a * 8 + 8)
        partials.update(find_a(prog, range_a))
best_a = min(solutions)
best_a

164279024971453

In [59]:
# double-check:
sol_prog = Program(best_a,0,0,prog)
while True:
    sol_prog.execute()

2,4,1,1,7,5,1,5,4,0,5,5,0,3,3,0,

IndexError: Program halted

That's the right answer! You are one gold star closer to finding the Chief Historian.

You have completed Day 17! You can [Share] this victory or [Return to Your Advent Calendar].